In [ ]:
!pip install pyngrok

In [ ]:
!python -m pip install streamlit


## 추천 시스템 UI

## library 및 드라이브 연결

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install lightfm-next

In [ ]:
!pip install streamlit pyngrok pyarrow fastparquet -q

In [ ]:
import streamlit as st
import numpy as np
import pandas as pd
import pickle
import scipy.sparse as sp
from datetime import datetime
import lightfm
from lightfm import LightFM
import sqlite3
from pathlib import Path

## 공통 경로



In [ ]:
%%writefile config.py
from pathlib import Path

# LightFM 모델 및 데이터 저장 경로
BASE_DIR = "/content/drive/MyDrive/lightfm_recsys/final"

# SQLite DB 파일 경로
DB_PATH = f"{BASE_DIR}/app.db"

def ensure_base_dir():
    """BASE_DIR가 없으면 생성."""
    Path(BASE_DIR).mkdir(parents=True, exist_ok=True)

Overwriting config.py


## DB 코드

In [ ]:
%%writefile db_utils.py
import sqlite3
from datetime import datetime
import pandas as pd

from config import DB_PATH, ensure_base_dir


def get_connection():
    """SQLite 커넥션 생성"""
    ensure_base_dir()
    conn = sqlite3.connect(DB_PATH, check_same_thread=False)
    return conn


def init_db():
    """DB 파일 및 테이블 생성"""
    conn = get_connection()
    cur = conn.cursor()

    # 사용자 테이블
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS users (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            email TEXT UNIQUE NOT NULL,
            password TEXT NOT NULL,
            name TEXT,
            lightfm_user_id INTEGER,
            created_at TEXT
        );
        """
    )

    # 피드백 테이블 (like / dislike)
    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS feedback (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            user_id INTEGER,          -- users.id (로그인한 유저, 없을 수도 있음)
            lightfm_user_id INTEGER,  -- 추천에 사용된 LightFM UserID
            product_id INTEGER,
            action TEXT,              -- 'like' / 'dislike'
            timestamp TEXT
        );
        """
    )

    conn.commit()
    conn.close()


def create_user(email, password, name, lightfm_user_id):
    """회원가입용 유저 생성"""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute(
        """
        INSERT INTO users (email, password, name, lightfm_user_id, created_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        (email, password, name, lightfm_user_id, datetime.now().isoformat(timespec="seconds")),
    )
    conn.commit()
    conn.close()


def find_user(email, password=None):
    """유저 조회 (password 없으면 이메일만으로, 있으면 로그인용)"""
    conn = get_connection()
    cur = conn.cursor()

    if password is None:
        cur.execute(
            "SELECT id, email, name, lightfm_user_id FROM users WHERE email = ?",
            (email,),
        )
    else:
        cur.execute(
            "SELECT id, email, name, lightfm_user_id FROM users WHERE email = ? AND password = ?",
            (email, password),
        )

    row = cur.fetchone()
    conn.close()

    if row:
        return {
            "id": row[0],
            "email": row[1],
            "name": row[2],
            "lightfm_user_id": row[3],
        }
    return None


def insert_feedback_db(user_id_db, lightfm_user_id, product_id, action):
    """피드백을 DB에 저장"""
    conn = get_connection()
    cur = conn.cursor()
    cur.execute(
        """
        INSERT INTO feedback (user_id, lightfm_user_id, product_id, action, timestamp)
        VALUES (?, ?, ?, ?, ?)
        """,
        (user_id_db, lightfm_user_id, product_id, action,
         datetime.now().isoformat(timespec="seconds")),
    )
    conn.commit()
    conn.close()


def get_online_metrics_db(filter_user_id_db=None):
    """
    DB에 저장된 피드백 기준 온라인 지표 계산
    - filter_user_id_db 가 있으면 해당 계정 데이터만 필터링
    """
    conn = get_connection()
    if filter_user_id_db is None:
        df = pd.read_sql_query("SELECT * FROM feedback", conn)
    else:
        df = pd.read_sql_query(
            "SELECT * FROM feedback WHERE user_id = ?",
            conn,
            params=(filter_user_id_db,),
        )
    conn.close()

    if df.empty:
        return None

    total_actions = len(df)
    like_cnt = (df["action"] == "like").sum()
    dislike_cnt = (df["action"] == "dislike").sum()
    ctr_like = like_cnt / total_actions if total_actions > 0 else 0.0

    by_user = (
        df.groupby("lightfm_user_id")["action"]
        .value_counts(normalize=True)
        .rename("ratio")
        .reset_index()
        .query("action == 'like'")
        .sort_values("ratio", ascending=False)
    )

    return {
        "total_actions": total_actions,
        "like_cnt": like_cnt,
        "dislike_cnt": dislike_cnt,
        "ctr_like": ctr_like,
        "by_user": by_user,
    }

Overwriting db_utils.py


## 추천 모듈

In [ ]:
%%writefile recsys_utils.py
import pickle
import numpy as np
import pandas as pd
import scipy.sparse as sp
import streamlit as st

from config import BASE_DIR


@st.cache_resource
def load_artifacts():
    """LightFM 모델 및 관련 데이터 로드"""
    # LightFM 모델
    with open(f"{BASE_DIR}/lightfm_model.pkl", "rb") as f:
        model = pickle.load(f)

    # 전체 interactions (user x item)
    interactions_all = sp.load_npz(f"{BASE_DIR}/interactions_all.npz")

    # grouped_sample
    grouped_sample = pd.read_parquet(f"{BASE_DIR}/grouped_sample.parquet")

    # items (index=ProductID)
    items = pd.read_parquet(f"{BASE_DIR}/items.parquet")
    if "ProductID" in items.columns:
        items = items.set_index("ProductID")

    return model, interactions_all, grouped_sample, items


def get_random_user_id(grouped_df: pd.DataFrame) -> int:
    """grouped_sample에서 임의의 UserID 하나 뽑기"""
    return int(grouped_df["UserID"].sample(1).values[0])


def get_user_history(grouped_df: pd.DataFrame, user_id: int, fill_na: bool = True):
    """
    특정 UserID에 대한 히스토리 (카테고리/브랜드별 event_strength 합계)와
    raw history를 함께 반환.
    """
    cols = ["product_id", "event_strength", "category_code", "brand"]
    history = grouped_df.query("UserID == @user_id")[cols].copy()

    if history.empty:
        return pd.DataFrame(), pd.DataFrame()

    if fill_na:
        history["category_code"] = history["category_code"].fillna("N/A")
        history["brand"] = history["brand"].fillna("N/A")

    history_agg = (
        history
        .groupby(["category_code", "brand"], as_index=False)["event_strength"]
        .sum()
        .sort_values("event_strength", ascending=False)
    )

    return history_agg, history


def lightfm_recommend(
    model,
    interactions,
    items_df,
    grouped_df,
    user_id: int,
    n: int = 10,
    fill_na: bool = True,
):
    """
    LightFM의 predict를 이용하여 Top-N 추천 아이템 반환.
    - interactions: (user x item) CSR matrix
    - items_df: index = ProductID (내부 item index와 동일하다고 가정)
    """

    n_users, n_items = interactions.shape

    # user_id가 내부 인덱스 범위 안에 있다는 가정
    if not (0 <= user_id < n_users):
        raise ValueError(f"user_id {user_id} is out of range (0 ~ {n_users - 1})")

    # 이미 상호작용한 아이템 파악
    user_interactions = interactions.tocsr()[user_id]
    known_items = user_interactions.indices

    # LightFM에서 점수 계산
    item_ids = np.arange(n_items, dtype=np.int32)
    user_ids = np.full(n_items, user_id, dtype=np.int32)

    scores = model.predict(user_ids, item_ids)

    # 이미 본 아이템은 점수 낮게 주기
    scores[known_items] = -1e9

    # Top-N
    top_items = np.argsort(-scores)[:n]

    # 메타 정보 조인
    rec_df = items_df.loc[top_items].copy()
    rec_df = rec_df.reset_index().rename(columns={"index": "ProductID"})

    if fill_na:
        if "category_code" in rec_df.columns:
            rec_df["category_code"] = rec_df["category_code"].fillna("N/A")
        if "brand" in rec_df.columns:
            rec_df["brand"] = rec_df["brand"].fillna("N/A")

    cols_order = ["ProductID", "product_id", "category_code", "brand"]
    rec_df = rec_df[[c for c in cols_order if c in rec_df.columns]]

    return rec_df

Overwriting recsys_utils.py


## streamlit 코드

In [ ]:
%%writefile app.py
import streamlit as st
import numpy as np
import pandas as pd
from datetime import datetime

from recsys_utils import (
    load_artifacts,
    get_random_user_id,
    get_user_history,
    lightfm_recommend
)
from db_utils import (
    init_db,
    create_user,
    find_user,
    insert_feedback_db,
    get_online_metrics_db,   # 안 써도 일단 그대로 둠
    get_connection
)

# DB 초기화 + LightFM 아티팩트 로드
init_db()
model, interactions_all, grouped_sample, items = load_artifacts()


# 세션 상태 초기화
if "feedback_logs" not in st.session_state:
    st.session_state["feedback_logs"] = []

if "selected_user_id" not in st.session_state:
    st.session_state["selected_user_id"] = None

if "user" not in st.session_state:
    # 로그인한 계정 정보 (users 테이블 row)
    st.session_state["user"] = None


def log_feedback(user_id: int, product_id: int, action: str):
    """
    세션 메모리 + SQLite DB 둘 다에 피드백 기록
    - user_id: LightFM UserID
    """
    # 세션 메모리
    st.session_state["feedback_logs"].append(
        {
            "user_id": user_id,
            "product_id": product_id,
            "action": action,
            "timestamp": datetime.now().isoformat(timespec="seconds"),
        }
    )

    # DB에 저장 (로그인 X >> user_id_db = None)
    user_db_id = (
        st.session_state["user"]["id"]
        if st.session_state.get("user") is not None
        else None
    )
    insert_feedback_db(user_db_id, user_id, product_id, action)


def get_online_metrics_session():
    """
    현재 세션에서 쌓인 feedback 기준 온라인 지표 계산
    - action: 'view' / 'cart' / 'purchase'
    - cart + purchase를 '클릭' 개념으로 간주
    """
    logs = st.session_state["feedback_logs"]
    if not logs:
        return None

    df = pd.DataFrame(logs)

    total_actions = len(df)
    view_cnt = (df["action"] == "view").sum()
    cart_cnt = (df["action"] == "cart").sum()
    purchase_cnt = (df["action"] == "purchase").sum()

    # cart 또는 purchase를 'click'으로 정의
    click_mask = df["action"].isin(["cart", "purchase"])
    ctr_click = click_mask.mean() if total_actions > 0 else 0.0

    # 유저별 click 비율 (cart 또는 purchase 비율)
    by_user = (
        df.assign(is_click=click_mask)
        .groupby("user_id")["is_click"]
        .mean()
        .reset_index()
        .rename(columns={"is_click": "click_rate"})
        .sort_values("click_rate", ascending=False)
    )

    return {
        "total_actions": total_actions,
        "view_cnt": view_cnt,
        "cart_cnt": cart_cnt,
        "purchase_cnt": purchase_cnt,
        "ctr_click": ctr_click,
        "by_user": by_user,
    }


# Streamlit UI
st.set_page_config(
    page_title="LightFM 추천 시스템",
    layout="wide",
)

st.title("LightFM 기반 상품 추천 시스템")

# -------- 사이드바: 계정 / 로그인 영역 --------
st.sidebar.header("계정 / 로그인")

# 회원가입
with st.sidebar.expander("회원가입", expanded=False):
    st.write("새 계정을 생성합니다.")

    signup_email = st.text_input("이메일", key="signup_email")
    signup_name = st.text_input("이름", key="signup_name")
    signup_pw = st.text_input("비밀번호", type="password", key="signup_pw")
    signup_pw2 = st.text_input("비밀번호 확인", type="password", key="signup_pw2")

    if not grouped_sample.empty:
        min_u = int(grouped_sample["UserID"].min())
        max_u = int(grouped_sample["UserID"].max())
    else:
        min_u, max_u = 0, 0

    signup_lfm_id = st.number_input(
        "LightFM UserID (내부 UserID 매핑용)",
        min_value=min_u,
        max_value=max_u if max_u > min_u else min_u,
        value=min_u,
        step=1,
        key="signup_lfm_id",
    )

    if st.button("회원가입 완료"):
        if signup_pw != signup_pw2:
            st.error("비밀번호가 일치하지 않습니다.")
        elif signup_email == "":
            st.error("이메일을 입력해 주세요.")
        else:
            try:
                if find_user(signup_email) is not None:
                    st.error("이미 존재하는 이메일입니다.")
                else:
                    create_user(
                        signup_email,
                        signup_pw,
                        signup_name,
                        int(signup_lfm_id),
                    )
                    st.success("회원가입이 완료되었습니다. 로그인 해 주세요.")
            except Exception as e:
                st.error(f"회원가입 중 오류 발생: {e}")

# 로그인
with st.sidebar.expander("로그인", expanded=True):
    if st.session_state["user"] is None:
        login_email = st.text_input("이메일", key="login_email")
        login_pw = st.text_input("비밀번호", type="password", key="login_pw")

        if st.button("로그인"):
            user = find_user(login_email, login_pw)
            if user is None:
                st.error("이메일 또는 비밀번호가 올바르지 않습니다.")
            else:
                st.session_state["user"] = user
                st.success(f"{user['name']}님, 환영합니다!")
    else:
        cur_user = st.session_state["user"]
        st.success(f"현재 로그인: {cur_user['name']} ({cur_user['email']})")

        if st.button("로그아웃"):
            st.session_state["user"] = None

st.sidebar.markdown("---")

# -------- 사이드바: 추천용 LightFM UserID 선택 --------
st.sidebar.header("추천용 사용자 선택")

user_select_mode = st.sidebar.radio(
    "LightFM UserID 선택 방식",
    options=["로그인 계정의 UserID 사용", "랜덤 사용자", "수동 입력"],
)

if user_select_mode == "로그인 계정의 UserID 사용":
    if st.session_state["user"] is None:
        st.info("로그인 후에 사용할 수 있습니다.")
        selected_user_id = None
    else:
        selected_user_id = int(st.session_state["user"]["lightfm_user_id"])

elif user_select_mode == "랜덤 사용자":
    if st.sidebar.button("🎲 랜덤 유저 뽑기"):
        st.session_state["selected_user_id"] = get_random_user_id(grouped_sample)
    selected_user_id = st.session_state["selected_user_id"]

else:  # "수동 입력"
    min_user = int(grouped_sample["UserID"].min())
    max_user = int(grouped_sample["UserID"].max())
    default_user = min_user

    user_input = st.sidebar.number_input(
        f"UserID (범위: {min_user} ~ {max_user})",
        min_value=min_user,
        max_value=max_user,
        value=default_user,
        step=1,
    )
    if st.sidebar.button("이 UserID로 설정"):
        st.session_state["selected_user_id"] = int(user_input)

    selected_user_id = st.session_state["selected_user_id"]

# 추천 개수
topn = st.sidebar.slider(
    "추천 개수 (Top-N)", min_value=5, max_value=50, value=10, step=5
)

st.sidebar.markdown("---")
st.sidebar.markdown("※ 피드백은 세션 메모리 + SQLite DB에 함께 저장됩니다.")


# 메인 레이아웃: 왼쪽(탭/추천) + 오른쪽(DB 성능 요약 패널)
col_main, col_perf = st.columns([4, 1])

# ---------------- 왼쪽: 탭 / 추천 / 로그 / DB 상세 ----------------
with col_main:
    # -------- 탭 구성 --------
    tab_demo, tab_metrics, tab_logs, tab_metrics_db = st.tabs(
        ["추천 데모", "세션 온라인 지표", "세션 피드백 로그", "DB 기반 온라인 지표"]
    )

    # ========== 탭 1: 추천 데모 ==========
    with tab_demo:
        st.subheader("1️⃣ 사용자 히스토리 & 추천 결과")

        user_id = selected_user_id

        if user_id is None:
            st.info("왼쪽 사이드바에서 LightFM UserID를 먼저 선택해 주세요.")
        else:
            st.markdown(f"**현재 사용 중인 LightFM UserID:** `{user_id}`")

            # 히스토리 & 추천
            history_agg, history_raw = get_user_history(grouped_sample, user_id)
            try:
                rec_df = lightfm_recommend(
                    model=model,
                    interactions=interactions_all,
                    items_df=items,
                    grouped_df=grouped_sample,
                    user_id=user_id,
                    n=topn,
                )
            except ValueError as e:
                st.error(str(e))
                rec_df = pd.DataFrame()

            if history_agg.empty:
                st.warning("이 유저의 히스토리가 없습니다. 다른 유저를 선택해 보세요.")
            else:
                col1, col2 = st.columns(2)

                with col1:
                    st.markdown("### 🧾 User History (카테고리/브랜드별 이벤트 강도)")
                    st.dataframe(history_agg, use_container_width=True, height=350)

                with col2:
                    st.markdown("### ⭐ 추천 결과")
                    if rec_df.empty:
                        st.warning("추천 결과가 없습니다.")
                    else:
                        st.dataframe(rec_df, use_container_width=True, height=350)

                st.markdown("---")
                st.markdown("### 👀 / 🛒 / 💳 추천 결과에 대한 피드백")

                if rec_df is not None and not rec_df.empty:
                    for _, row in rec_df.iterrows():
                        p_id = int(row["ProductID"])
                        product_name = row.get("product_id", "")
                        cat = row.get("category_code", "")
                        brand = row.get("brand", "")

                        # 상품 정보 + 3개 버튼 (view / cart / purchase)
                        cols = st.columns([4, 1, 1, 1])

                        with cols[0]:
                            st.markdown(
                                f"**ProductID:** `{p_id}`  |  "
                                f"상품: `{product_name}`  |  "
                                f"카테고리: `{cat}`  |  브랜드: `{brand}`"
                            )

                        # 1) 약한 positive: view
                        with cols[1]:
                            if st.button("👀 View", key=f"view_{user_id}_{p_id}"):
                                log_feedback(user_id=user_id, product_id=p_id, action="view")

                        # 2) 중간 positive: cart
                        with cols[2]:
                            if st.button("🛒 Cart", key=f"cart_{user_id}_{p_id}"):
                                log_feedback(user_id=user_id, product_id=p_id, action="cart")

                        # 3) 강한 positive: purchase
                        with cols[3]:
                            if st.button("💳 Purchase", key=f"purchase_{user_id}_{p_id}"):
                                log_feedback(user_id=user_id, product_id=p_id, action="purchase")

    # ========== 탭 2: 세션 기준 온라인 지표 ==========
    with tab_metrics:
        st.subheader("2️⃣ 세션 기준 온라인 성능 지표")

        metrics = get_online_metrics_session()
        if metrics is None:
            st.info("아직 남겨진 피드백이 없습니다. 추천 탭에서 버튼을 눌러보세요.")
        else:
            st.markdown(
                f"""
                - 총 피드백 수: **{metrics['total_actions']}**
                - 👀 view 수: **{metrics['view_cnt']}**
                - 🛒 cart 수: **{metrics['cart_cnt']}**
                - 💳 purchase 수: **{metrics['purchase_cnt']}**
                - cart/purchase 기준 클릭률: **{metrics['ctr_click']:.2%}**
                """
            )

            st.markdown("#### LightFM UserID별 click 비율 (세션 기준)")
            st.dataframe(metrics["by_user"], use_container_width=True)

    # ========== 탭 3: 세션 피드백 로그 ==========
    with tab_logs:
        st.subheader("3️⃣ 세션 내 피드백 Raw Log")

        logs = st.session_state["feedback_logs"]
        if not logs:
            st.info("현재 세션에 저장된 피드백이 없습니다.")
        else:
            log_df = pd.DataFrame(logs)
            st.dataframe(log_df, use_container_width=True)

    # ========== 탭 4: DB 기반 온라인 지표 ==========
    with tab_metrics_db:
        st.subheader("4️⃣ DB 기반 온라인 성능 지표 (세션과 무관, 누적 기준)")

        if st.session_state["user"] is not None:
            only_me = st.checkbox("현재 로그인한 계정의 데이터만 보기", value=False)
        else:
            only_me = False

        user_db_id = (
            st.session_state["user"]["id"]
            if (st.session_state["user"] is not None and only_me)
            else None
        )

        # DB에서 직접 feedback 읽어서 view/cart/purchase 기준으로 지표 계산
        conn = get_connection()
        if user_db_id is None:
            df_db = pd.read_sql_query(
                "SELECT * FROM feedback ORDER BY id DESC",
                conn
            )
        else:
            df_db = pd.read_sql_query(
                "SELECT * FROM feedback WHERE user_id = ? ORDER BY id DESC",
                conn,
                params=(user_db_id,),
            )
        conn.close()

        if df_db.empty:
            st.info("DB에 저장된 피드백이 아직 없습니다.")
        else:
            total_actions_db = len(df_db)
            view_cnt_db = (df_db["action"] == "view").sum()
            cart_cnt_db = (df_db["action"] == "cart").sum()
            purchase_cnt_db = (df_db["action"] == "purchase").sum()

            click_mask_db = df_db["action"].isin(["cart", "purchase"])
            ctr_click_db = click_mask_db.mean() if total_actions_db > 0 else 0.0

            # 최근 10개 기준 클릭률
            last10 = df_db.head(10)
            last10_click_rate = (
                last10["action"].isin(["cart", "purchase"]).mean()
                if len(last10) > 0 else 0.0
            )

            # LightFM UserID별 click rate
            by_lfm_user = (
                df_db.assign(is_click=click_mask_db)
                .groupby("lightfm_user_id")["is_click"]
                .mean()
                .reset_index()
                .rename(columns={"is_click": "click_rate"})
                .sort_values("click_rate", ascending=False)
            )

            st.markdown(
                f"""
                - 총 피드백 수: **{total_actions_db}**
                - 👀 view 수: **{view_cnt_db}**
                - 🛒 cart 수: **{cart_cnt_db}**
                - 💳 purchase 수: **{purchase_cnt_db}**
                - cart/purchase 기준 전체 클릭률: **{ctr_click_db:.2%}**
                - 최근 10개 기준 클릭률(cart/purchase): **{last10_click_rate:.2%}**
                """
            )

            st.markdown("#### LightFM UserID별 click 비율 (DB 기준)")
            st.dataframe(by_lfm_user, use_container_width=True)

            st.markdown("#### Raw 피드백 데이터 (최신순)")
            st.dataframe(df_db, use_container_width=True, height=300)

# ---------------- 오른쪽: 항상 보이는 DB 기준 성능 요약 패널 ----------------
with col_perf:
    st.markdown("### 📊 DB 기준 성능 요약")

    conn = get_connection()
    df_perf = pd.read_sql_query(
        "SELECT * FROM feedback ORDER BY id DESC",
        conn
    )
    conn.close()

    if df_perf.empty:
        st.write("피드백 데이터 없음")
    else:
        total_actions_perf = len(df_perf)
        view_cnt_perf = (df_perf["action"] == "view").sum()
        cart_cnt_perf = (df_perf["action"] == "cart").sum()
        purchase_cnt_perf = (df_perf["action"] == "purchase").sum()

        click_mask_perf = df_perf["action"].isin(["cart", "purchase"])
        ctr_click_perf = click_mask_perf.mean() if total_actions_perf > 0 else 0.0

        # 숫자 카드 형태로 핵심 지표 표시
        st.metric("전체 action 수", total_actions_perf)
        st.metric("cart/purchase 클릭률", f"{ctr_click_perf:.1%}")

        # action 비율 막대그래프
        action_ratio = (
            df_perf["action"]
            .value_counts(normalize=True)
            .reindex(["view", "cart", "purchase"])
            .fillna(0.0)
        )

        st.markdown("#### action 비율")
        st.bar_chart(action_ratio)

        # 최근 10개 기준 클릭률
        last10_perf = df_perf.head(10)
        last10_ctr_perf = (
            last10_perf["action"].isin(["cart", "purchase"]).mean()
            if len(last10_perf) > 0 else 0.0
        )
        st.markdown(f"최근 10개 클릭률(cart/purchase): **{last10_ctr_perf:.1%}**")

Overwriting app.py


In [ ]:
!streamlit run app.py --server.port 8501 &>/content/logs.txt &


In [ ]:
from pyngrok import ngrok
ngrok.set_auth_token("35xgNqbkovX5fvj6tAKgluXtSg0_6rCY95osm8N8jt5MZFYvD")
public_url = ngrok.connect(8501)
public_url

<NgrokTunnel: "https://michal-trickier-laurie.ngrok-free.dev" -> "http://localhost:8501">

In [ ]:
import time
while True:
    time.sleep(60)